# Classes, files, and the traps that bite

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~35 min

## Classes: state plus behaviour

A **class** bundles data (the **attributes**, `self.x`) with the functions that act on it (**methods**). `__init__` runs when you write `Averager()` and sets the attributes up. `self` is just "this object" — Python passes it to every method automatically.



In [ ]:
class Averager:
    def __init__(self):              # runs at Averager()
        self.total, self.n = 0.0, 0
    def update(self, value):         # self = the object it was called on
        self.total += value
        self.n += 1
    @property                        # lets us write avg.mean — no ()
    def mean(self):
        return self.total / max(self.n, 1)   # max(): avoid ÷0 before any update

avg = Averager()
avg.update(0.8); avg.update(0.6)
avg.mean                       # 0.7

This is not decoration: a PyTorch `nn.Module` model, a `Dataset`, a sklearn estimator — all are classes where `__init__` does setup and methods (`forward`, `fit`, `predict`) do the work. Subclassing `nn.Module` in AI1 is exactly this pattern.



## Reading and writing files

`with open(path) as fh` opens a file and **closes it automatically** when the block ends — even if an error occurs. Iterating the file object yields lines one at a time, lazily: a 10 GB log doesn't load into RAM. `"w"` writes (and truncates first!), `"a"` appends.



In [ ]:
with open("scores.txt", "w", encoding="utf-8") as fh:
    fh.write("0.91\n")            # write() needs the \n added yourself
    fh.write("0.87\n")

with open("scores.txt", encoding="utf-8") as fh:
    vals = [float(line) for line in fh]  # iterate lines; float() strips the \n
vals                               # [0.91, 0.87]

`encoding="utf-8"` is cheap insurance against mangled characters on other machines. For paths, `pathlib.Path` beats string surgery: `Path("data") / "train.csv"` — the `/` joins path parts (an overloaded operator, not division) — builds a path that works on Windows and Linux alike.



## JSON and CSV: the two contest formats

Tables live in CSV (pandas reads them — next lesson); configs, annotations and label maps live in **JSON**. `json.dumps` serializes a dict to a string, `json.loads` parses it back; `dump`/`load` do the same with a file object.



In [ ]:
import json
meta = {"seed": 42, "classes": ["cat", "dog"]}
json.dumps(meta)                 # '{"seed": 42, "classes": ["cat", "dog"]}'
json.loads('{"seed": 42}')       # {'seed': 42}

with open("run.json", "w") as fh:
    json.dump(meta, fh, indent=2)     # pretty-printed into the file

- JSON knows only dict/list/str/int/float/bool/None — tuples come back as lists, sets don't serialize at all, NumPy types need `.item()` first.
- Pickle/joblib can store arbitrary Python objects (fitted models), but never `pickle.load` a file you didn't create — unpickling executes embedded code.



## Stdlib helpers worth knowing



In [ ]:
from collections import Counter, defaultdict
Counter("mississippi")           # Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
labels = ["cat", "dog", "cat", "bird", "dog", "cat"]
Counter(labels).most_common(2)   # [('cat', 3), ('dog', 2)] — the top-k answer

groups = defaultdict(list)       # a missing key creates an empty list, no setdefault
groups["a"].append(1)

In [ ]:
import math, random, time, itertools
math.log2(8)                     # 3.0 — cross-entropy lives on logs
xs = [3, 1, 2]
random.seed(0); random.shuffle(xs)
itertools.chain([1, 2], [3])     # lazily yields 1, 2, 3 — flatten nested lists
t0 = time.time()                 # time.time() - t0 = a cell's wall-clock seconds

`try/except` catches an expected failure and continues — useful when 3 rows in 100 000 are corrupt:



In [ ]:
lines = ["0.9", "0.8", "oops", "0.7"]
good, bad = [], 0
for line in lines:
    try:
        good.append(float(line))
    except ValueError:               # catch the SPECIFIC error you expect
        bad += 1
good, bad                        # ([0.9, 0.8, 0.7], 1)

> **Note.** A bare `except:` swallows everything, including Ctrl-C and real bugs. Name the exception type and count what you skipped — silent data loss fails submissions quietly.



## Defensive habits that save contests

- **Assert the contract**: `assert len(pred) == len(test)`, `assert set(pred) <= {0, 1}` (on sets `<=` means 'is a subset of') — fail loudly at the bug, not at the grader.
- Print shapes and a `head()` after every transform — 30 seconds of printing catches axis slips and bad merges before they corrupt the model.
- Check the trivial case by hand: if your pipeline can't handle 5 samples, it won't handle the dataset.
- Restart-and-run-all before trusting a notebook number — stale state is the most common fake result.



### ✏️ Why wrap file access in `with open(...) as fh:`?

**A.** It encrypts the contents as UTF-8
**B.** It reads the file in parallel threads for speed
**C.** It closes the file automatically, even on error
**D.** It is only needed for JSON files

<details><summary>Answer</summary>

**C.** It closes the file automatically, even on error

The with-block guarantees close() on exit — no leaked handles, no half-written files.

</details>

### ✏️ int(s) fails on one bad line in a loop. Which handler is right?

**A.** except Everything:
**B.** finally:
**C.** except:
**D.** except ValueError:

<details><summary>Answer</summary>

**D.** except ValueError:

Catch the specific error you expect; a bare except also hides KeyboardInterrupt and real bugs.

</details>

---

**Go deeper**
- [The Python Tutorial (official)](https://docs.python.org/3/tutorial/)
- [PyTorch: Learn the Basics](https://pytorch.org/tutorials/beginner/basics/intro.html)